In [ ]:
import os
import numpy as np
import pandas as pd
from PyLTSpice import SimRunner, SpiceEditor, RawRead

# --- Configuration ---
NETLIST_FILE = "NC_schematic_real_world_model.net" 

# Define the range of resistance values to test for Rtune (in Ohms)
# Sweeping from 1kΩ to 20kΩ in steps of 100Ω
r_values = np.arange(1000, 20000, 100) 

# Dictionary to store the frequency responses
simulation_results = {}

print("Starting LTspice automated sweeps...")

# Initialize the simulator runner
runner = SimRunner()

for r in r_values:
    # Load the netlist into the editor for this iteration
    editor = SpiceEditor(NETLIST_FILE)
    
    # Target the Rtune resistor and change its value. 
    editor.set_component_value('Rtune', str(r)) 
    
    # Run the simulation 
    task = runner.run(editor)
    
    # 2. Wait for LTspice to finish
    raw_file, log_file = task.wait_results()
    
    # 3. SAFETY CHECK: Stop the crash and read the error!
    if raw_file is None:
        print("\nCRITICAL ERROR: LTspice aborted!")
        print(f"Check this log file for the exact SPICE error: {log_file}")
        break 
        
    # 4. Only read if the file actually exists
    raw_data = RawRead(raw_file)
    freqs = raw_data.get_trace('frequency').get_wave()
    
    # Extract the voltage at the cancellation node.
    # .ac analysis outputs complex numbers, so we take the absolute value (magnitude)
    v_out_complex = raw_data.get_trace('V(canceledout)').get_wave()
    v_out_magnitude = np.abs(v_out_complex)
    
    # Store the results for this specific resistance
    simulation_results[r] = v_out_magnitude

print("Simulations complete. Processing data...")

optimal_dataset = []

# We iterate through the frequency array to find the best Rtune for each specific frequency
for i, freq in enumerate(freqs):
    # Convert complex frequency to real number
    f_real = np.real(freq)
    
    # We only care about the audio frequency range where cancellation matters (e.g., 100Hz to 3kHz)
    if 100 <= f_real <= 3000:
        min_amplitude = float('inf')
        best_r = None
        
        # Check every resistor value to see which one produced the lowest amplitude at this frequency
        for r in r_values:
            current_amplitude = simulation_results[r][i]
            
            if current_amplitude < min_amplitude:
                min_amplitude = current_amplitude
                best_r = r
                
        optimal_dataset.append({
            "Frequency_Hz": round(f_real, 2),
            "Optimal_Rtune_Ohms": best_r,
            "Min_Noise_Amplitude_V": min_amplitude
        })

# --- Save to CSV ---
df = pd.DataFrame(optimal_dataset)
csv_filename = "anc_training_dataset.csv"
df.to_csv(csv_filename, index=False)

print(f"Dataset successfully generated! Saved to '{csv_filename }'.")
print(df.head(10))

Starting LTspice automated sweeps...
Simulations complete. Processing data...
Dataset successfully generated! Saved to 'anc_training_dataset.csv'.
   Frequency_Hz  Optimal_Rtune_Ohms  Min_Noise_Amplitude_V
0        102.33                5000               0.000009
1        104.71                5000               0.000009
2        107.15                5000               0.000009
3        109.65                5000               0.000008
4        112.20                5000               0.000008
5        114.82                5000               0.000007
6        117.49                5000               0.000007
7        120.23                5000               0.000006
8        123.03                5000               0.000005
9        125.89                5000               0.000005


In [ ]:
import joblib
import pandas as pd

# 1. Load the pre-trained model from the .pkl file
print("Booting up ANC AI...")
try:
    anc_ai = joblib.load('anc_tuning_model.pkl')
    print("Model loaded successfully!")
except FileNotFoundError:
    print("Error: Could not find the .pkl file.")
    exit()

# 2. Define a function to get the tuning resistance
def get_optimal_resistance(frequency):
    """
    Takes an incoming noise frequency and returns the exact 
    resistance needed to phase-shift and cancel it.
    """
    # The model expects a DataFrame with the exact column name used in training
    input_data = pd.DataFrame([[frequency]], columns=['Frequency_Hz'])
    
    # Predict and extract the first (and only) value
    predicted_resistance = anc_ai.predict(input_data)[0]
    
    return predicted_resistance


incoming_freq = 1000  

print(f"\n[MIC INPUT] Detected dominant frequency: {incoming_freq} Hz")

optimal_r = get_optimal_resistance(incoming_freq)

print(f"[AI OUTPUT] Tuning circuit... Set Rtune to: {optimal_r:.2f} Ohms")

Booting up ANC AI...
Model loaded successfully!

[MIC INPUT] Detected dominant frequency: 1000 Hz
[AI OUTPUT] Tuning circuit... Set Rtune to: 5200.00 Ohms
